## Bayes Institution

In [ ]:

# 得病的概率，（先验概率）
PROB_SICK = 0.0001

# 检测的准确率
PROB_POSITIVE_GIVEN_SICK = 0.99
## 这里假设了 生病检出阳性 和 健康检出阴性 的概率是相等的
PROB_POSITIVE_GIVEN_HEALTHY = (1 - PROB_POSITIVE_GIVEN_SICK)

"""
Deduction

PROB_SICK_IN_POSITIVE =
    (P(SICK and POSITIVE)) / P(POSITIVE)
PROB_POSITIVE_IN_SICK = 
    (P(POSITIVE and SICK)) / P(SICK)
PROB_SICK_AND_POSITIVE =
    PROB_SICK_IN_POSITIVE * P(POSITIVE) =
    PROB_POSITIVE_IN_SICK * P(SICK)

SO...
PROB_SICK_IN_POSITIVE =
    PROB_POSITIVE_IN_SICK * P(SICK) / P(POSITIVE)

"""

PROB_POSITIVE = PROB_POSITIVE_GIVEN_SICK * PROB_SICK +PROB_POSITIVE_GIVEN_HEALTHY * (1 - PROB_SICK)

PROB_SICK_IN_POSITIVE = PROB_POSITIVE_GIVEN_SICK * PROB_SICK / PROB_POSITIVE

print(f"检测准确率99%，得病概率0.01%，检测呈阳性，得病概率是多少？")
print(f"得病概率：{PROB_SICK_IN_POSITIVE:.4f}")


检测准确率99%，得病概率0.01%，检测呈阳性，得病概率是多少？
得病概率：0.0098


### 贝叶斯：糖果袋的故事

想象你有一个**不透明的糖果袋**。

袋子里大多是**巧克力**，偶尔有一颗**草莓糖**。  
你摸出一颗，**没看**，先舔一下：甜甜的、有点酸 → 好像是草莓味。

问：这颗**真的是草莓糖**的可能有多大？

你脑子里其实在做三件事：

1. **以前就知道的事（先验）**  
   袋子里草莓糖本来就很少。  
   → 还没尝之前的猜测。

2. **刚刚尝到的味道（似然）**  
   如果真是草莓糖，很容易尝出酸甜；  
   如果是巧克力，偶尔也可能有点怪味道。  
   → 这个味道，在「是草莓」时有多常见。

3. **合在一起想一想（后验）**  
   「本来就少」+「味道有点像」→ 更新后的想法。  
   → 尝过之后你有多相信是草莓糖。

一句话：**先有一个猜测，看到一点线索，再把猜测改一改。**

公式其实就在说这个：

$$
\text{尝完后有多信}
= \frac{\text{真是草莓时尝到这味道有多正常} \times \text{尝之前有多信}}{\text{不管是啥糖，尝到这味道有多常见}}
$$

分母的意思是：巧克力也可能骗你一口，要把这种「假线索」也算进去。

---

### 和上面「患病检测」一一对应

| 糖果故事 | 患病检测（上面的代码） | 符号 |
|---------|------------------------|------|
| 这颗是草莓糖 | 你真的得病了 | \(A\) / SICK |
| 尝到酸甜味 | 检测呈阳性 | \(B\) / POSITIVE |
| 袋子里草莓本来就少 | 得病本来就罕见（0.01%） | \(P(A)\) 先验 |
| 真是草莓时，很常尝出酸甜 | 有病时，99% 测出阳性 | \(P(B\|A)\) 似然 |
| 巧克力偶尔也像草莓味 | 没病时，也可能假阳性 | \(P(B\|\neg A)\) |
| 尝完后：多相信是草莓？ | 阳性后：多相信有病？ | \(P(A\|B)\) 后验 |

所以上面算出「阳性后得病概率只有约 0.98%」并不奇怪：

- 检测很准（味道很像草莓）→ 似然很大  
- 但病很少见（袋子里草莓本来就少）→ 先验很小  
- 健康人更多，假阳性也会贡献不少「阳性」→ 分母把信念稀释了  

**贝叶斯 = 用新线索，轻轻推一下旧想法。**  
线索再强，如果旧想法是「这件事极少发生」，更新后也不一定变得很肯定。


## Naive Bayes

```
P(class | feature_1, feature_2, ..., feature_n)
  = P(class) * P(feature_1|class) * P(feature_2|class) * ... * P(feature_n|class)
    / P(feature_1, feature_2, ..., feature_n)
```

In [5]:
# 玩具邮件：词是否出现（1/0）。Naive = 假设词之间条件独立。
# 词汇：free, money, meeting

# 训练集 (free, money, meeting) -> spam?
emails = [
    ((1, 1, 0), True),   # "free money"
    ((1, 0, 0), True),   # "free offer"
    ((0, 1, 0), True),   # "win money"
    ((0, 0, 1), False),  # "team meeting"
    ((0, 0, 0), False),  # "hello"
    ((1, 0, 1), False),  # "free meeting slot"
]

WORDS = ("free", "money", "meeting")
n_spam = sum(1 for _, is_spam in emails if is_spam)
n_ham = len(emails) - n_spam

# 先验
p_spam = n_spam / len(emails)
p_ham = n_ham / len(emails)

# P(word=1 | class)，Laplace 平滑：避免训练集没出现过的词把概率打成 0
def word_prob(word_idx, is_spam_class):
    count = sum(
        features[word_idx]
        for features, label in emails
        if label == is_spam_class
    )
    n_class = n_spam if is_spam_class else n_ham
    return (count + 1) / (n_class + 2)


# 新邮件："free money now" -> (free=1, money=1, meeting=0)
new_email = (1, 1, 0)


def likelihood(features, is_spam_class):
    """P(features | class) ≈ ∏ P(word_i | class)  （条件独立假设）"""
    p = 1.0
    for i, present in enumerate(features):
        p_word = word_prob(i, is_spam_class)
        p *= p_word if present else (1 - p_word)
    return p


# 后验 ∝ 先验 × 似然；分母 P(features) 用全概率归一化
score_spam = p_spam * likelihood(new_email, True)
score_ham = p_ham * likelihood(new_email, False)
p_features = score_spam + score_ham

p_spam_given = score_spam / p_features
p_ham_given = score_ham / p_features

print(f"新邮件特征 {dict(zip(WORDS, new_email))}")
print(f"P(spam)={p_spam:.3f}, P(ham)={p_ham:.3f}")
print(f"P(spam | email) = {p_spam_given:.4f}")
print(f"P(ham  | email) = {p_ham_given:.4f}")
print(f"判定: {'spam' if p_spam_given > p_ham_given else 'ham'}")


新邮件特征 {'free': 1, 'money': 1, 'meeting': 0}
P(spam)=0.500, P(ham)=0.500
P(spam | email) = 0.9000
P(ham  | email) = 0.1000
判定: spam


## Build your Own

In [ ]:
def bayes(prior, likelihood, false_positive_rate):
    evidence = likelihood * prior + false_positive_rate * (1 - prior)
    posterior = likelihood * prior / evidence
    return posterior

print(bayes(PROB_SICK, PROB_POSITIVE_GIVEN_SICK, PROB_POSITIVE_GIVEN_HEALTHY))

0.00980392156862745


In [ ]:
import math

from collections import defaultdict

class NaiveBayes:
    def __init__(self, smooting = 1.0):
        self.smooting = smooting
        self.class_counts = defaultdict(int)
        self.word_counts = defaultdict(lambda: defaultdict(int))
        self.class_word_totals = defaultdict(int)
        self.vocab = set()

    def train(self, documents, labels):
        for doc, label in zip(documents, labels):
            self.class_counts[label] += 1
            words = doc.lower().split()
            for word in words:
                self.word_counts[label][word] += 1
                self.class_word_totals[label] += 1
                self.vocab.add(word)

    def predict(self, document):
        words = document.lower().split()
        total_docs = sum(self.class_counts.values())
        vocab_size = len(self.vocab)
        best_class = None
        best_score = float("-inf")
        for cls in self.class_counts:
            score = math.log(self.class_counts[cls] / total_docs)
            for word in words:
                count = self.word_counts[cls].get(word, 0)
                total = self.class_word_totals[cls]
                score += math.log((count + self.smooting) / (total + self.smooting * vocab_size))
            if score > best_score:
                best_score = score
                best_class = cls
        return best_class


In [8]:
train_docs = [
    "win free money now",
    "free lottery ticket winner",
    "claim your prize today free",
    "urgent offer free cash",
    "congratulations you won free",
    "meeting tomorrow at noon",
    "project update attached",
    "can we schedule a call",
    "quarterly report review",
    "lunch on thursday sounds good",
    "team standup notes attached",
    "please review the pull request",
]

train_labels = [
    "spam", "spam", "spam", "spam", "spam",
    "ham", "ham", "ham", "ham", "ham", "ham", "ham",
]

classifier = NaiveBayes()
classifier.train(train_docs, train_labels)

test_messages = [
    "free money waiting for you",
    "meeting rescheduled to friday",
    "you won a free prize",
    "please review the attached report",
]

for msg in test_messages:
    print(f"  '{msg}' -> {classifier.predict(msg)}")

  'free money waiting for you' -> spam
  'meeting rescheduled to friday' -> spam
  'you won a free prize' -> spam
  'please review the attached report' -> ham


In [9]:
def show_top_words(classifier, cls, n = 5):
    vocab_size = len(classifier.vocab)
    total = classifier.class_word_totals[cls]
    probs = {}

    for word in classifier.vocab:
        count = classifier.word_counts[cls].get(word, 0)
        probs[word] = (count + classifier.smooting) / (total + classifier.smooting * vocab_size)
    sorted_words = sorted(probs.items(), key=lambda x: x[1], reverse=True)
    for word, prob in sorted_words[:n]:
        print(f"   {word}: {prob:.4f}")

print("\n Top spam words: ")
show_top_words(classifier, "spam")

print("\n Top ham words: ")
show_top_words(classifier, "ham")


 Top spam words: 
   free: 0.0923
   lottery: 0.0308
   win: 0.0308
   money: 0.0308
   ticket: 0.0308

 Top ham words: 
   attached: 0.0411
   review: 0.0411
   quarterly: 0.0274
   standup: 0.0274
   the: 0.0274


## Monte Carlo

In [ ]:
"""
A: Beta(1 + 50, 1 + 950)
B: Beta(1 + 65, 1 + 935)
"""